# Conway's Game of Life
### Lesson 4, Section 1

In this notebook we will:
1. Understand the rules of Conway's Game of Life
2. Implement the core functions **step by step**
3. Visualise the evolution of patterns
4. Explore still lifes, oscillators, and spaceships
5. Build a complete simulation with animation
6. Experiment with different initial conditions

---

## 0 · Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from IPython.display import clear_output
import time

In [ ]:
# Colour scheme
DEAD_COLOR = np.array([238, 243, 106]) / 255
LIVE_COLOR = np.array([63, 48, 71]) / 255
GOL_CMAP = ListedColormap([DEAD_COLOR, LIVE_COLOR])

def show_grid(grid, title='', ax=None):
    """Display a Game of Life grid."""
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(grid, cmap=GOL_CMAP, vmin=0, vmax=1)
    ax.set_title(title, fontweight='bold')
    ax.axis('off')
    return ax

---
## 1 · The Rules

Conway's Game of Life operates on a 2D grid where each cell is either **alive** (1) or **dead** (0).

At each time step, every cell is updated **simultaneously** based on its **8 neighbours** (Moore neighbourhood):

| # | Rule | Effect |
|---|------|--------|
| 1 | Live cell with **< 2** live neighbours | Dies (underpopulation) |
| 2 | Live cell with **2 or 3** live neighbours | Survives |
| 3 | Live cell with **> 3** live neighbours | Dies (overpopulation) |
| 4 | Dead cell with **exactly 3** live neighbours | Becomes alive (reproduction) |

> These four simple rules produce an astonishing variety of complex behaviour.

---
## 2 · Step 1: Counting Neighbours

The first building block is a function that counts how many **live neighbours** a cell has.

Each cell is surrounded by up to **8 cells** — horizontally, vertically, and diagonally. This arrangement is called the **Moore neighbourhood**. To count live neighbours we slice out the 3×3 patch centred on the cell, sum all values, then subtract the cell's own state (so a cell does not count itself as its own neighbour).

**Boundary handling:** cells on the grid edge have fewer than 8 neighbours. The simplest approach is to clip the slice to the valid index range, effectively treating out-of-bounds positions as permanently dead. An alternative is a *toroidal* (wrap-around) grid, where the top edge connects to the bottom and the left edge to the right; this is explored in Section 7.


In [ ]:
def count_neighbours(row, col, grid):
    """Count the number of alive neighbours of cell (row, col).
    
    Uses the Moore neighbourhood (8 surrounding cells).
    Handles boundary cells by clipping to grid edges.
    """
    height, width = grid.shape
    # Extract neighbourhood (clipped at boundaries)
    neighbourhood = grid[
        max(0, row - 1) : min(height, row + 2),
        max(0, col - 1) : min(width, col + 2)
    ]
    # Sum all cells in neighbourhood, subtract the cell itself
    return int(np.sum(neighbourhood)) - int(grid[row, col])

### Test: Count neighbours

Let's verify with a small grid:

In [ ]:
test_grid = np.array([
    [0, 1, 0],
    [1, 1, 0],
    [0, 0, 1]
])

show_grid(test_grid, title='Test grid (3×3)')
plt.show()

# Centre cell (1,1) is alive with neighbours at (0,1), (1,0), (2,2) → 3 neighbours
print(f'Neighbours of (1,1): {count_neighbours(1, 1, test_grid)}  (expected: 3)')
# Corner cell (0,0) is dead, has neighbours (0,1)=1, (1,0)=1, (1,1)=1 → 3 neighbours
print(f'Neighbours of (0,0): {count_neighbours(0, 0, test_grid)}  (expected: 3)')
# Cell (0,2) has neighbours (0,1)=1, (1,1)=1, (1,2)=0 → 2 neighbours
print(f'Neighbours of (0,2): {count_neighbours(0, 2, test_grid)}  (expected: 2)')

---
## 3 · Step 2: Updating a Single Cell

Given a cell's current state and its neighbour count, we apply the four Conway rules to determine its **next state**.

An important subtlety: every cell is evaluated based on the *current* grid — not one that is being partially updated. This **synchronous (parallel) update** is what distinguishes cellular automata from sequential models. If we updated cells one at a time and immediately used the new values, the result would depend on the order of traversal and would be a different model.

In [ ]:
def update_cell(is_alive, neighbours_alive):
    """Determine the next state of a cell.
    
    Parameters
    ----------
    is_alive : int (0 or 1)
        Current state of the cell.
    neighbours_alive : int
        Number of alive neighbours.
    
    Returns
    -------
    int : next state (0 or 1)
    """
    if is_alive:
        # Rules 1-3: survival requires exactly 2 or 3 neighbours
        return 1 if neighbours_alive in (2, 3) else 0
    else:
        # Rule 4: birth requires exactly 3 neighbours
        return 1 if neighbours_alive == 3 else 0

### Test: Update cell

In [ ]:
# Alive cell with 1 neighbour → dies (underpopulation)
print(f'Alive, 1 neighbour: {update_cell(1, 1)}  (expected: 0)')
# Alive cell with 2 neighbours → survives
print(f'Alive, 2 neighbours: {update_cell(1, 2)}  (expected: 1)')
# Alive cell with 4 neighbours → dies (overpopulation)
print(f'Alive, 4 neighbours: {update_cell(1, 4)}  (expected: 0)')
# Dead cell with 3 neighbours → born
print(f'Dead, 3 neighbours: {update_cell(0, 3)}  (expected: 1)')
# Dead cell with 2 neighbours → stays dead
print(f'Dead, 2 neighbours: {update_cell(0, 2)}  (expected: 0)')

---
## 4 · Step 3: One Full Time Step

We iterate over every cell, compute its neighbour count from the *current* grid, and write the result into a *separate new grid*. Only after all cells are evaluated do we replace the old grid with the new one.

In [ ]:
def game_of_life_step(grid):
    """Advance the Game of Life by one time step.
    
    All cells are updated simultaneously (parallel update).
    """
    height, width = grid.shape
    new_grid = np.zeros_like(grid)
    for row in range(height):
        for col in range(width):
            nbrs = count_neighbours(row, col, grid)
            new_grid[row, col] = update_cell(int(grid[row, col]), nbrs)
    return new_grid

---
## 5 · Classic Patterns

Thousands of patterns have been catalogued. They fall into a few broad categories:

| Category | Behaviour | Examples |
|----------|-----------|---------|
| **Still life** | Does not change between steps | Block, Beehive, Loaf |
| **Oscillator** | Returns to its initial state after a fixed number of steps (*period*) | Blinker (period 2), Pulsar (period 3) |
| **Spaceship** | Translates across the grid while cycling through states | Glider, Lightweight spaceship |
| **Methuselah** | Small pattern that takes many steps to stabilise | R-pentomino (1103 steps), Acorn |

### 5a · Still Life — Block

A 2×2 block is the simplest still life. Each of the four live cells has exactly **3 live neighbours** (the other three corners), satisfying the survival rule. Each dead cell adjacent to the block has at most **2 live neighbours**, so no new cells are born. The configuration is therefore perfectly stable.



In [ ]:
def place_pattern(grid, pattern, position):
    """Place a pattern on the grid at the given (row, col) position."""
    h, w = pattern.shape
    r, c = position
    grid[r:r+h, c:c+w] = pattern
    return grid

block = np.array([[1, 1],
                  [1, 1]])

grid = np.zeros((8, 8), dtype=int)
place_pattern(grid, block, (3, 3))

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for i, ax in enumerate(axes):
    show_grid(grid, title=f't = {i}', ax=ax)
    grid = game_of_life_step(grid)
plt.suptitle('Still Life: Block', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()
print('The block does not change — it is a still life.')

### 5b · Oscillator — Blinker
A horizontal bar of 3 cells oscillates between horizontal and vertical.

In [ ]:
blinker = np.array([[1, 1, 1]])

grid = np.zeros((8, 8), dtype=int)
place_pattern(grid, blinker, (3, 3))

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for i, ax in enumerate(axes):
    show_grid(grid, title=f't = {i}', ax=ax)
    grid = game_of_life_step(grid)
plt.suptitle('Oscillator: Blinker (period 2)', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()
print('The blinker alternates between horizontal and vertical — period 2.')

### 5c · Spaceship — Glider
The glider is the smallest and most celebrated spaceship. It was discovered by **Richard Guy** in 1970 while Conway's group was systematically searching for patterns with long-lasting or travelling behaviour. It translates **one cell diagonally every 4 steps** — a propagation speed of c/4, where c is the theoretical maximum (one cell per step).

Gliders have a big role in Game of Life: a sustained stream of gliders can encode binary signals, and by arranging glider guns and deflectors it is possible to construct logic gates. Essentailly people are builing compupters within the game and gliders are a big part of it 

In [ ]:
glider = np.array([
    [0, 1, 0],
    [0, 0, 1],
    [1, 1, 1]
])

grid = np.zeros((20, 20), dtype=int)
place_pattern(grid, glider, (1, 1))

fig, axes = plt.subplots(1, 5, figsize=(16, 3))
for i in range(5):
    show_grid(grid, title=f't = {i*4}', ax=axes[i])
    for _ in range(4):
        grid = game_of_life_step(grid)
plt.suptitle('Spaceship: Glider (moves diagonally)', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()
print('The glider translates one cell diagonally every 4 steps.')

---
## 6 · Full Simulation with Animation

We now run the model on a large random grid and observe both the spatial evolution and the population count over time.

**What to expect:** starting from random noise, the population typically passes through three phases:

1. **Chaotic phase** — rapid, unpredictable turnover as overcrowded clusters collapse and isolated cells die off
2. **Consolidation** — the population drops sharply as patterns settle into still lifes and low-period oscillators
3. **Quasi-stable phase** — a slow, long-term drift driven by occasional gliders colliding with still lifes and generating new transients

The **initial density** (fraction of live cells at t = 0) strongly shapes this trajectory. Very low or very high densities tend to collapse quickly; densities around 0.3–0.4 produce the richest long-term dynamics.

In [ ]:
# Parameters
AREA = 60
INITIAL_DENSITY = 0.3
STEPS = 100
FRAME_DELAY = 0.05  # seconds between frames

# Random initial grid
np.random.seed(42)
grid = (np.random.random((AREA, AREA)) < INITIAL_DENSITY).astype(int)

# Track population over time
population = []

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

for t in range(STEPS):
    population.append(np.sum(grid))
    
    ax1.clear()
    ax1.imshow(grid, cmap=GOL_CMAP, vmin=0, vmax=1)
    ax1.set_title(f'Game of Life — Step {t}', fontweight='bold')
    ax1.axis('off')
    
    ax2.clear()
    ax2.plot(population, color=LIVE_COLOR, lw=2)
    ax2.set_xlabel('Time step')
    ax2.set_ylabel('Live cells')
    ax2.set_title('Population', fontweight='bold')
    ax2.set_xlim(0, STEPS)
    ax2.set_ylim(0, max(population) * 1.2)
    
    clear_output(wait=True)
    display(fig)
    time.sleep(FRAME_DELAY)
    
    grid = game_of_life_step(grid)

plt.close(fig)
print(f'Simulation complete. Final population: {population[-1]} cells.')

---
## 7 · Faster Implementation with NumPy

The cell-by-cell loop is slow. We can use **convolution** to count all neighbours at once:

In [ ]:
from scipy.signal import convolve2d

KERNEL = np.array([[1, 1, 1],
                   [1, 0, 1],
                   [1, 1, 1]])

def game_of_life_step_fast(grid):
    """Vectorised Game of Life step using convolution."""
    neighbours = convolve2d(grid, KERNEL, mode='same', boundary='fill', fillvalue=0)
    # Birth: dead cell with exactly 3 neighbours
    birth = (grid == 0) & (neighbours == 3)
    # Survival: alive cell with 2 or 3 neighbours
    survive = (grid == 1) & ((neighbours == 2) | (neighbours == 3))
    return (birth | survive).astype(int)

In [ ]:
# Verify: both implementations give the same result
np.random.seed(99)
test = (np.random.random((20, 20)) < 0.3).astype(int)
slow_result = game_of_life_step(test)
fast_result = game_of_life_step_fast(test)

assert np.array_equal(slow_result, fast_result), 'Mismatch!'
print('Both implementations produce identical results.')

In [ ]:
# Timing comparison
big_grid = (np.random.random((100, 100)) < 0.3).astype(int)

import timeit
t_slow = timeit.timeit(lambda: game_of_life_step(big_grid), number=3) / 3
t_fast = timeit.timeit(lambda: game_of_life_step_fast(big_grid), number=3) / 3

print(f'Slow (loops):      {t_slow:.4f} s')
print(f'Fast (convolution): {t_fast:.4f} s')
print(f'Speedup: {t_slow / t_fast:.0f}x')

---
## 8 · What to Try Next

Try modifying the code above to explore:

1. **Different initial patterns**: Create and place a [Gosper glider gun](https://conwaylife.com/wiki/Gosper_glider_gun) or an [R-pentomino](https://conwaylife.com/wiki/R-pentomino).
2. **Grid size**: What happens with a very small grid (10×10) vs. a very large one (200×200)?
3. **Initial density**: Sweep density from 0.1 to 0.9 and plot the final population. Is there a critical density?
4. **Boundary conditions**: Modify the fast implementation to use `boundary='wrap'` (toroidal world). How does this change behaviour?
5. **Population dynamics**: Track the population over 1000 steps. Does it converge, oscillate, or keep changing?

---

## 9 · Mini-Projects

### Project A: Pattern Library
Build a dictionary of classic patterns (block, blinker, glider, LWSS, pulsar, Gosper gun). Write a function that places any pattern at any position and orientation. Run them all and classify each as still life, oscillator, or spaceship.

### Project B: Statistical Study
Run 50 simulations with random initial conditions at different densities. For each, record: (a) how many steps until the population stabilises, (b) the final population, (c) the number of distinct still lifes and oscillators. Plot these statistics.

### Project C: Modified Rules
Experiment with alternative rule sets (e.g., HighLife: B36/S23, or Day & Night: B3678/S34678). Compare the types of structures that emerge.